# RAGTime: PDF Chat with RAG

A simple implementation of Retrieval-Augmented Generation for finding relevant content, or "chatting with your PDFs".

This notebook:
1. Processes PDFs from your Google Drive folder
2. Creates embeddings for semantic search
3. Enables natural language queries with cited responses

Perfect for: Searching a custom knowledge base, literature reviews, meeting notes summarization, and annual report writing.

Requirements:
- OpenAI API key
- PDFs in your specified Google Drive folder

## Instructions
1. Provide OpenAI API key and GDrive folder path in this first cell below.
2. Run the "Code Setup" cell.
3. Run the "Process pdfs" cell.
4. Search and Chat!

In [11]:
#################################
# Provide OpenAI API Key
#################################
from google.colab import userdata

# Enter your OpenAI API Key into the CoLab "secrets" section (key icon at left)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# alt (worse) enter it here as a variable and uncomment this line:
# OPENAI_API_KEY = 'key-string-123'


#################################
# Provide Path to GDrive folder
#################################

# Specify the folder path in your Google Drive
folder_path = '/content/drive/MyDrive/32\.\ Ilkmaar/Ilkmaar_RAG'  # Replace RAG_TEST with the path to your folder

# 1. Code Setup

### 1. Install and Import

In [2]:
!pip install neo4j
!pip install boto3
!pip install dotenv
!pip install httpx==0.27.2
!pip install openai==1.55.3
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install -qU langchain-openai
!pip install langchain_community
!pip install -qU pypdf
!pip install -qU langchain-unstructured
!pip install tiktoken python-dotenv
!pip install PyPDF2

import os
import io
from uuid import uuid4
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from enum import Enum
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import HumanMessage, SystemMessage
import PyPDF2
from PyPDF2 import PdfReader
import json
import openai
from openai import OpenAI
from google.colab import userdata
from google.colab import drive
from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.10.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.72.0
    Uninstalling open

### 2. Define and set up chat completions

In [5]:
openai.api_key = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)

# (Optional) edit the model
OPENAI_CHAT_MODEL = "gpt-4.1"

DEFAULT_PROMPT = "You are a helpful assistant. Use the provided context to answer the user's question."
DEFAULT_RESPONSE_NUMBER = 3

class Citation(BaseModel):
    author: str
    year: int

class AnswerWithCitations(BaseModel):
    answer: str
    citations: List[Citation]

DEFAULT_SCHEMA = AnswerWithCitations

In [6]:
def find_relevant(query: str, n: int = DEFAULT_RESPONSE_NUMBER):
    # Perform similarity search
    results = vectorstore.similarity_search(query, n)
    retrieved_chunks = [doc.page_content for doc in results]
    return retrieved_chunks

def answer_query(query: str, prompt: str = DEFAULT_PROMPT, schema: BaseModel = DEFAULT_SCHEMA, n=DEFAULT_RESPONSE_NUMBER) -> BaseModel:
    relevant_chunks = find_relevant(query, n)
    context_str = "\n".join(relevant_chunks)

    completion = client.beta.chat.completions.parse(
        model=OPENAI_CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": f"{prompt}"
            },
            {
                "role": "user",
                "content": f"Context:\n{context_str}\n\nQuestion: {query}",
            },
        ],
        tools=[
            openai.pydantic_function_tool(schema),
        ],
    )

    tool_call = (completion.choices[0].message.tool_calls or [])[0]
    parsed_args = tool_call.function.parsed_arguments
    assert isinstance(parsed_args, AnswerWithCitations)

    return parsed_args

# 2. Process pdfs

In [7]:
# Optional: configure the default chunk size (characters) and overlap
DEFAULT_CHUNK_SIZE = 2000
DEFAULT_CHUNK_OVERLAP = 200

In [8]:
# Mount Google Drive
drive.mount('/content/drive')

if not os.path.exists(folder_path):
    print(f"Error: Folder '{folder_path}' not found.")
else:
    # Get a list of PDF files in the folder
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]

# Process each PDF file
for pdf_file in pdf_files:
    pdf_path = os.path.join(folder_path, pdf_file)
    with open(pdf_path, 'rb') as f:
        pdf_content = f.read()

        reader = PdfReader(io.BytesIO(pdf_content))

        # Extract text
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        # Split text into chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=DEFAULT_CHUNK_SIZE, chunk_overlap=DEFAULT_CHUNK_OVERLAP)
        chunks = text_splitter.split_text(text)

# Set up embeddings and vector store
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vectorstore = InMemoryVectorStore(embeddings)

# Embed chunks and add to vector store
for chunk in chunks:
    vectorstore.add_texts([chunk], metadatas=[{"id": str(uuid4())}])

Mounted at /content/drive


<ipython-input-8-78fa85a37ae4>:30: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)


# Search and Chat!

## Demo 1: Search for relevant pdf sections

Edit the query string below, then run the cell.

(Optional) The number of results to include can be configured in the setup above.

In [10]:
query = "Why is it important for girls to learn data science?"

results = find_relevant(query)
results

['The Isles of Ilkmaar: A Data-rich, Multiplayer Virtual World to Support Middle School Girls’ Interest in Data Science  This three-year Innovations in Development project led by the Concord Consortium in partnership with the University of Miami and FableVision Games aims to promote middle school-aged girls’ interest and aspirations in data science and data-rich futures, through an identity-aligned, social game-based learning approach powered by a data-rich, multiplayer virtual world.  Ensuring equitable representation in data literacy and data science careers is critical for the future. Data science is a rapidly growing field at the interdisciplinary intersection of computing and statistics. Yet even though women make up almost 40% of academic statisticians, they hold only about 15% of positions in data science (Burtch, 2018). In particular, Latina women are most drastically underrepresented in data science, relative to the U.S. population, occupying only 2% of the computing workforce

## Demo 2: Chat

Edit the query string below, then run the cell.

(Optional) The prompt, number of RAG results, and response format can be configured in the setup above.

In [ ]:
# Ask your question
query = "What will the project do in Year 1?"

# Perform the query and get the response
response = answer_query(query)

# Access the returned values:
print("Answer:\n")
print(response.answer)
print("\n")
print("Citations:")
for c in response.citations:
    print(f"- {c.author} ({c.year})")

Answer:

In Year 1, the project will focus on Phase 1 of research, which includes initial game design and development. Specifically, the team will recruit and run four participatory co-design groups consisting of six girls each, totaling 24 participants. Over two three-hour weekend sessions per group, the girls will play the initial prototype of the game while providing feedback. Researchers will conduct focus group discussions and engage participants in design and ideation tasks to gather insights on compelling game narratives, activities, and the useful forms of data for these goals. The co-design sessions will be collaboratively planned by the research team, FableVision, and project advisors, leveraging expertise in user testing, interviews related to STEM learning and identity, and participatory co-design processes. The analyses of data from these sessions will help ensure that the game enables girls to use data for personal and social goals aligned with their identities.


Citatio